In [22]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import spacy as spacy

RUTA_PROYECTO = Path("..")
RUTA_provisional = RUTA_PROYECTO / "data" / "provisional"
RUTA_procesado = RUTA_PROYECTO / "data" / "procesado"
RUTA_NORMALIZADOS = RUTA_procesado / "resenas_normalizadas"

carpeta_resenas = (
    RUTA_provisional / "resenas_videojuegos_bloques"
)


archivos_resenas = sorted(
    carpeta_resenas.glob("*.parquet")
)


archivos_normalizados = sorted(
    RUTA_NORMALIZADOS.glob(
        "resenas_normalizadas_*.parquet"
    )
)

print(
    "Archivos normalizados encontrados:",
    len(archivos_normalizados)
)


Archivos normalizados encontrados: 47


In [23]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

print("Modelo cargado:", nlp.meta["name"])
print("Idioma:", nlp.meta["lang"])

Modelo cargado: core_web_sm
Idioma: en


In [24]:
import sys
import spacy
import pandas as pd
from pathlib import Path

print("Intérprete:", sys.executable)

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

print("Modelo cargado:", nlp.meta["name"])
print("Idioma:", nlp.meta["lang"])

Intérprete: c:\Users\Paula María\Desktop\IS\5º CURSO\TFG\Trabajo-Fin-de-Grado\.venv\Scripts\python.exe
Modelo cargado: core_web_sm
Idioma: en


In [25]:
# Comprobaión lematización y palabras vacias
texto_prueba = (
    "I didn't like the games, "
    "but the graphics were amazing!"
)

documento = nlp(texto_prueba)

print("Comprobación lematización y detección de palabras vacías:")

for token in documento:
    print(
        token.text,
        "| lema:", token.lemma_,
        "| palabra vacía:", token.is_stop,
        "| puntuación:", token.is_punct
    )

Comprobación lematización y detección de palabras vacías:
I | lema: I | palabra vacía: True | puntuación: False
did | lema: do | palabra vacía: True | puntuación: False
n't | lema: not | palabra vacía: True | puntuación: False
like | lema: like | palabra vacía: False | puntuación: False
the | lema: the | palabra vacía: True | puntuación: False
games | lema: game | palabra vacía: False | puntuación: False
, | lema: , | palabra vacía: False | puntuación: True
but | lema: but | palabra vacía: True | puntuación: False
the | lema: the | palabra vacía: True | puntuación: False
graphics | lema: graphic | palabra vacía: False | puntuación: False
were | lema: be | palabra vacía: True | puntuación: False
amazing | lema: amazing | palabra vacía: False | puntuación: False
! | lema: ! | palabra vacía: False | puntuación: True


In [26]:
# Comprobacion negaciones

textos_negacion = [
    "This game is good.",
    "This game is not good.",
    "I don't recommend this game.",
    "I never enjoyed playing it.",
    "The graphics aren't impressive."
]

for texto in textos_negacion:
    documento = nlp(texto)

    print("\nTexto:", texto)

    for token in documento:
        print(
            token.text,
            "| lema:", token.lemma_,
            "| palabra vacía:", token.is_stop
        )


Texto: This game is good.
This | lema: this | palabra vacía: True
game | lema: game | palabra vacía: False
is | lema: be | palabra vacía: True
good | lema: good | palabra vacía: False
. | lema: . | palabra vacía: False

Texto: This game is not good.
This | lema: this | palabra vacía: True
game | lema: game | palabra vacía: False
is | lema: be | palabra vacía: True
not | lema: not | palabra vacía: True
good | lema: good | palabra vacía: False
. | lema: . | palabra vacía: False

Texto: I don't recommend this game.
I | lema: I | palabra vacía: True
do | lema: do | palabra vacía: True
n't | lema: not | palabra vacía: True
recommend | lema: recommend | palabra vacía: False
this | lema: this | palabra vacía: True
game | lema: game | palabra vacía: False
. | lema: . | palabra vacía: False

Texto: I never enjoyed playing it.
I | lema: I | palabra vacía: True
never | lema: never | palabra vacía: True
enjoyed | lema: enjoy | palabra vacía: False
playing | lema: play | palabra vacía: False
it | 

In [27]:
negaciones_conservadas = {
    "no",
    "not",
    "n't",
    "never",
    "neither",
    "nor"
}

In [28]:
# Definición de funcion para tokenizr correctamente con negaciones
def tokenizar_lematizar(texto):
    """
    Tokeniza y lematiza una reseña.

    Elimina puntuación, espacios y palabras vacías,
    pero conserva las negaciones por su importancia
    en el análisis de sentimiento.
    """

    documento = nlp(texto)
    lemas = []

    for token in documento:

        # Ignorar espacios y signos de puntuación
        if token.is_space or token.is_punct:
            continue

        lema = token.lemma_.lower().strip()

        # Comprobar la forma original y el lema porque
        # spaCy transforma n't en not
        es_negacion = (
            token.text.lower() in negaciones_conservadas
            or lema in negaciones_conservadas
        )

        # Eliminar stopwords, excepto las negaciones
        if token.is_stop and not es_negacion:
            continue

        # Evitar añadir lemas vacíos
        if lema:
            lemas.append(lema)

    return " ".join(lemas)


In [29]:
textos_prueba = [
    "This game is good.",
    "This game is not good.",
    "I don't recommend this game.",
    "I never enjoyed playing it.",
    "The graphics aren't impressive."
]

for texto in textos_prueba:
    print("Texto original:")
    print(texto)

    print("Texto procesado:")
    print(tokenizar_lematizar(texto))

    print("-" * 50)

Texto original:
This game is good.
Texto procesado:
game good
--------------------------------------------------
Texto original:
This game is not good.
Texto procesado:
game not good
--------------------------------------------------
Texto original:
I don't recommend this game.
Texto procesado:
not recommend game
--------------------------------------------------
Texto original:
I never enjoyed playing it.
Texto procesado:
never enjoy play
--------------------------------------------------
Texto original:
The graphics aren't impressive.
Texto procesado:
graphic not impressive
--------------------------------------------------


In [30]:
# Comprobar lematización en bloque de prueba

bloque_prueba = pd.read_parquet(
    archivos_normalizados[0]
)

print(
    "Reseñas en el bloque:",
    f"{len(bloque_prueba):,}"
)

muestra_prueba = (
    bloque_prueba
    .head(100)
    .copy()
)

print(
    "Reseñas seleccionadas:",
    len(muestra_prueba)
)

muestra_prueba ["review_text_processed"] = (
    muestra_prueba["review_text_normalized"]
    .apply(tokenizar_lematizar)
)


# Comprobación
muestra_prueba[
    [
        "review_text_normalized",
        "review_text_processed"
    ]
].head(20)



Reseñas en el bloque: 39,175
Reseñas seleccionadas: 100


,review_text_normalized,review_text_processed
0,it’s pretty sexual. not my fav i’m playing on ...,pretty sexual not fav play ps5 interesting uni...
1,good. a bit slow nostalgic fun. a bit slow. i ...,good bit slow nostalgic fun bit slow hope not ...
2,... an order for my kids & they have really en...,order kid enjoy play pc game order kid enjoy p...
3,what a great game! fun for everyone and keeps ...,great game fun keep move get game time son wii...
4,disney lion king 1.5 updating my disney movies...,disney lion king 1.5 update disney movie blu r...
5,love the udraw i bought this as a christmas pr...,love udraw buy christmas present granddaughter...
6,good chocolates husband bought these for me fo...,good chocolate husband buy 25th anniversary lo...
7,monopoly variety game for switch i purchased a...,monopoly variety game switch purchase switch a...
8,non stop fun i still own my wii which works pe...,non stop fun wii work perfectly love dance gam...
9,did not work for more than an hour stopped wor...,not work hour stop work hour son start play st...


In [31]:
# Revisar posibles vacíos 
mascara_vacios = (
    muestra_prueba["review_text_processed"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

muestra_prueba.loc[
    mascara_vacios,
    [
        "rating",
        "review_text",
        "review_text_normalized",
        "review_text_processed"
    ]
]

,rating,review_text,review_text_normalized,review_text_processed


In [32]:
carpeta_procesadas = (
    RUTA_procesado / "resenas_lematizadas"
)
if carpeta_procesadas.exists():
    archivos_anteriores = list(
        carpeta_procesadas.glob("*.parquet")
    )

    if archivos_anteriores:
        print(
            f"Se han encontrado {len(archivos_anteriores)} "
            "archivos anteriores."
        )

    else:
        print("La carpeta existe, pero no contiene archivos Parquet.")

else:
    carpeta_procesadas.mkdir(
        parents=True,
        exist_ok=True
    )

    print("La carpeta de salida no existía y ha sido creada.")





Se han encontrado 47 archivos anteriores.


In [33]:
def procesar_documento(documento):
    """
    Obtiene los lemas relevantes de un documento procesado por spaCy.

    Elimina espacios, puntuación y palabras vacías,
    excepto las negaciones.
    """

    lemas = []

    for token in documento:

        # Ignorar espacios y signos de puntuación
        if token.is_space or token.is_punct:
            continue

        lema = token.lemma_.lower().strip()

        # Comprobar si el token representa una negación
        es_negacion = (
            token.text.lower() in negaciones_conservadas
            or lema in negaciones_conservadas
        )

        # Eliminar palabras vacías excepto las negaciones
        if token.is_stop and not es_negacion:
            continue

        # Evitar lemas vacíos
        if lema:
            lemas.append(lema)

    return " ".join(lemas)

In [34]:
# Probar npl.pipe en bloque

#Seleccionar columna
textos_prueba = (
    muestra_prueba["review_text_normalized"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Procesar los textos
documentos = nlp.pipe(
    textos_prueba,
    batch_size=64
)

textos_procesados = [
    procesar_documento(documento)
    for documento in documentos
]

#Añadir resultados
muestra_prueba["review_text_processed"] = (
    textos_procesados
)

In [35]:
# Comprobar

muestra_prueba[
    [
        "review_text_normalized",
        "review_text_processed"
    ]
].head(10)

,review_text_normalized,review_text_processed
0,it’s pretty sexual. not my fav i’m playing on ...,pretty sexual not fav play ps5 interesting uni...
1,good. a bit slow nostalgic fun. a bit slow. i ...,good bit slow nostalgic fun bit slow hope not ...
2,... an order for my kids & they have really en...,order kid enjoy play pc game order kid enjoy p...
3,what a great game! fun for everyone and keeps ...,great game fun keep move get game time son wii...
4,disney lion king 1.5 updating my disney movies...,disney lion king 1.5 update disney movie blu r...
5,love the udraw i bought this as a christmas pr...,love udraw buy christmas present granddaughter...
6,good chocolates husband bought these for me fo...,good chocolate husband buy 25th anniversary lo...
7,monopoly variety game for switch i purchased a...,monopoly variety game switch purchase switch a...
8,non stop fun i still own my wii which works pe...,non stop fun wii work perfectly love dance gam...
9,did not work for more than an hour stopped wor...,not work hour stop work hour son start play st...


In [36]:
# Comprobar que no falte ningun registro
print(
    "Textos originales:",
    len(textos_prueba)
)

print(
    "Textos procesados:",
    len(textos_procesados)
)

print(
    "¿Coinciden?",
    len(textos_prueba) == len(textos_procesados)
)

Textos originales: 100
Textos procesados: 100
¿Coinciden? True


In [37]:
# Comprobar textosp procesados vacios

textos_vacios_prueba = (
    muestra_prueba["review_text_processed"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Textos procesados vacíos:", textos_vacios_prueba,
)

Textos procesados vacíos: 0


In [38]:
# Columnas que se conservarán en el conjunto lematizado
columnas_necesarias = [
    "rating",
    "parent_asin",
    "user_id",
    "timestamp",
    "verified_purchase",
    "review_text_normalized"
]

total_procesadas_ejecucion = 0
total_textos_vacios_ejecucion = 0

for numero_bloque, archivo in enumerate(
    archivos_normalizados,
    start=0
):
    ruta_salida = carpeta_procesadas / (
        f"resenas_lematizadas_{numero_bloque:04d}.parquet"
    )

    # No volver a procesar un bloque ya generado
    if ruta_salida.exists():
        print(
            f"Bloque {numero_bloque + 1}/"
            f"{len(archivos_normalizados)} ya existe. "
            "Se omite."
        )
        continue

    # Cargar solo las columnas necesarias
    bloque = pd.read_parquet(
        archivo,
        columns=columnas_necesarias
    )

    textos = (
        bloque["review_text_normalized"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    documentos = nlp.pipe(
        textos,
        batch_size=1000
    )

    bloque["review_text_processed"] = [
        procesar_documento(documento)
        for documento in documentos
    ]

    textos_vacios_bloque = (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    bloque.to_parquet(
        ruta_salida,
        index=False
    )

    total_procesadas_ejecucion += len(bloque)
    total_textos_vacios_ejecucion += textos_vacios_bloque

    print(
        f"Bloque {numero_bloque + 1}/"
        f"{len(archivos_normalizados)}: "
        f"{len(bloque):,} procesadas, "
        f"{textos_vacios_bloque:,} vacías"
    )

Bloque 1/47 ya existe. Se omite.
Bloque 2/47 ya existe. Se omite.
Bloque 3/47 ya existe. Se omite.
Bloque 4/47 ya existe. Se omite.
Bloque 5/47 ya existe. Se omite.
Bloque 6/47 ya existe. Se omite.
Bloque 7/47 ya existe. Se omite.
Bloque 8/47 ya existe. Se omite.
Bloque 9/47 ya existe. Se omite.
Bloque 10/47 ya existe. Se omite.
Bloque 11/47 ya existe. Se omite.
Bloque 12/47 ya existe. Se omite.
Bloque 13/47 ya existe. Se omite.
Bloque 14/47 ya existe. Se omite.
Bloque 15/47 ya existe. Se omite.
Bloque 16/47 ya existe. Se omite.
Bloque 17/47 ya existe. Se omite.
Bloque 18/47 ya existe. Se omite.
Bloque 19/47 ya existe. Se omite.
Bloque 20/47 ya existe. Se omite.
Bloque 21/47 ya existe. Se omite.
Bloque 22/47 ya existe. Se omite.
Bloque 23/47 ya existe. Se omite.
Bloque 24/47 ya existe. Se omite.
Bloque 25/47 ya existe. Se omite.
Bloque 26/47 ya existe. Se omite.
Bloque 27/47 ya existe. Se omite.
Bloque 28/47 ya existe. Se omite.
Bloque 29/47 ya existe. Se omite.
Bloque 30/47 ya existe.

In [39]:
print("\nRESULTADO DEL PROCESAMIENTO LINGÜÍSTICO")
print("---------------------------------------")
print(
    "Reseñas procesadas:",
    f"{total_procesadas_ejecucion:,}"
)
print(
    "Textos procesados vacíos:",
    f"{total_textos_vacios_ejecucion:,}"
)


RESULTADO DEL PROCESAMIENTO LINGÜÍSTICO
---------------------------------------
Reseñas procesadas: 0
Textos procesados vacíos: 0


In [40]:
# Revisar vacias

archivos_lematizados = sorted(
    carpeta_procesadas.glob(
        "resenas_lematizadas_*.parquet"
    )
)

resenas_procesadas_vacias = []

for archivo in archivos_lematizados:
    bloque = pd.read_parquet(
        archivo,
        columns=[
            "rating",
            "parent_asin",
            "review_text_normalized",
            "review_text_processed"
        ]
    )

    mascara_vacios = (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    seleccion = bloque.loc[
        mascara_vacios
    ].copy()

    if not seleccion.empty:
        resenas_procesadas_vacias.append(
            seleccion
        )


In [41]:
resenas_procesadas_vacias = pd.concat(
    resenas_procesadas_vacias,
    ignore_index=True
)

print(
    "Reseñas procesadas vacías:",
    f"{len(resenas_procesadas_vacias):,}"
)

resenas_procesadas_vacias.head(30)

Reseñas procesadas vacías: 220


,rating,parent_asin,review_text_normalized,review_text_processed
0,5,B07WLT1C27,a a,
1,4,B004RLL4XK,: :),
2,4,B084QDTVC6,none none,
3,5,B006HZA8EA,well well,
4,4,B07V5LZMTH,a a,
5,5,B07T85XHX6,a a,
6,5,B079Y44LDC,a a,
7,5,B087NM8BB6,a a,
8,5,B000SFK0SO,:) :),
9,5,B00XBLQCLQ,***** *****,


In [42]:
resenas_procesadas_vacias[
    [
        "rating",
        "review_text_normalized",
        "review_text_processed"
    ]
].head(50)

,rating,review_text_normalized,review_text_processed
0,5,a a,
1,4,: :),
2,4,none none,
3,5,well well,
4,4,a a,
5,5,a a,
6,5,a a,
7,5,a a,
8,5,:) :),
9,5,***** *****,


In [43]:
# Por qué es eliminado cada término

textos_diagnostico = [
    "a a",
    ":) :)",
    "none none",
    "well well",
    "***** *****",
    "get it just do it",
    "must have must have",
    "done must have",
    "top top"
]

for texto in textos_diagnostico:
    print("\nTEXTO:", repr(texto))

    documento = nlp(texto)

    for token in documento:
        print(
            repr(token.text),
            "| lema:", repr(token.lemma_),
            "| stopword:", token.is_stop,
            "| puntuación:", token.is_punct,
            "| espacio:", token.is_space,
            "| alfabético:", token.is_alpha
        )


TEXTO: 'a a'
'a' | lema: 'a' | stopword: True | puntuación: False | espacio: False | alfabético: True
'a' | lema: 'a' | stopword: True | puntuación: False | espacio: False | alfabético: True

TEXTO: ':) :)'
':)' | lema: ':)' | stopword: False | puntuación: True | espacio: False | alfabético: False
':)' | lema: ':)' | stopword: False | puntuación: True | espacio: False | alfabético: False

TEXTO: 'none none'
'none' | lema: 'none' | stopword: True | puntuación: False | espacio: False | alfabético: True
'none' | lema: 'none' | stopword: True | puntuación: False | espacio: False | alfabético: True

TEXTO: 'well well'
'well' | lema: 'well' | stopword: True | puntuación: False | espacio: False | alfabético: True
'well' | lema: 'well' | stopword: True | puntuación: False | espacio: False | alfabético: True

TEXTO: '***** *****'
'*' | lema: '*' | stopword: False | puntuación: True | espacio: False | alfabético: False
'*' | lema: '*' | stopword: False | puntuación: True | espacio: False | alfa

In [44]:
for texto in textos_diagnostico:
    print(
        repr(texto),
        "→",
        repr(tokenizar_lematizar(texto))
    )

'a a' → ''
':) :)' → ''
'none none' → ''
'well well' → ''
'***** *****' → ''
'get it just do it' → ''
'must have must have' → ''
'done must have' → ''
'top top' → ''


In [45]:
import re as re

emoticonos_positivos = {
    ":)",
    ":-)",
    "=)",
    ":D",
    ":-D",
    ";)",
    ";-)",
    "(:",
    ";p",
    ";-p",
    "<3"
}

emoticonos_negativos = {
    ":(",
    ":-(",
    "=(",
    ":'(",
    "):"
}

def convertir_emoticonos(texto):
    """
    Sustituye emoticonos positivos y negativos por marcadores textuales.
    También transforma grupos de estrellas en una señal positiva.
    """

    texto = str(texto)

    # Emoticonos positivos
    for emoticono in emoticonos_positivos:
        texto = re.sub(
            re.escape(emoticono),
            " positivo ",
            texto
    )

    # Emoticonos negativos
        for emoticono in emoticonos_negativos:
            texto = re.sub(
                re.escape(emoticono),
                " negativo ",
                texto
        )

    # Grupos de estrellas
    texto = re.sub(
        r"\*{3,}",
        " estrellas ",
        texto
    )

    return texto

In [46]:
print(convertir_emoticonos(":) :)"))
print(convertir_emoticonos("*****"))

 positivo   positivo 
 estrellas 


In [47]:
# Probar integración emoticonos con texto

textos_prueba = [
    "Great game :)",
    "Amazing :D",
    "I love it ;-)",
    "Bad game :(",
    "Very disappointing :-(",
    "*****",
    ":) :)",
    "I liked it, but the ending was bad :("
]

for texto in textos_prueba:
    print(
        repr(texto),
        "→",
        repr(convertir_emoticonos(texto))
    )

'Great game :)' → 'Great game  positivo '
'Amazing :D' → 'Amazing  positivo '
'I love it ;-)' → 'I love it  positivo '
'Bad game :(' → 'Bad game  negativo '
'Very disappointing :-(' → 'Very disappointing  negativo '
'*****' → ' estrellas '
':) :)' → ' positivo   positivo '
'I liked it, but the ending was bad :(' → 'I liked it, but the ending was bad  negativo '


In [48]:
# integración con el resto del texto 

def recuperar_texto_vacio(texto):
    """
    Recupera el contenido de una reseña que quedó vacía
    después del procesamiento lingüístico general.
    """

    # Convertir emoticonos y estrellas antes de usar spaCy
    texto = convertir_emoticonos(texto)

    documento = nlp(texto)

    lemas = []

    for token in documento:

        # Eliminar espacios y puntuación restante
        if token.is_space or token.is_punct:
            continue

        lema = token.lemma_.lower().strip()

        if lema:
            lemas.append(lema)

    return " ".join(lemas)

In [49]:
# Comprobación

textos_diagnostico = [
    "a a",
    ":) :)",
    "none none",
    "well well",
    "***** *****",
    "get it just do it",
    "must have must have",
    "done must have",
    "top top"
]

for texto in textos_diagnostico:
    print(
        repr(texto),
        "→",
        repr(recuperar_texto_vacio(texto))
    )

'a a' → 'a a'
':) :)' → 'positivo positivo'
'none none' → 'none none'
'well well' → 'well well'
'***** *****' → 'estrellas estrella'
'get it just do it' → 'get it just do it'
'must have must have' → 'must have must have'
'done must have' → 'do must have'
'top top' → 'top top'


In [50]:
# Aplicar recuperación 
total_textos_recuperados = 0
total_textos_vacios_finales = 0

for numero_bloque, archivo in enumerate(
    archivos_lematizados,
    start=1
):
    bloque = pd.read_parquet(archivo)

    # Identificar textos que quedaron vacíos
    mascara_vacios = (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    vacios_bloque = int(
        mascara_vacios.sum()
    )

    # Aplicar la recuperación solo a los vacíos
    if vacios_bloque > 0:
        bloque.loc[
            mascara_vacios,
            "review_text_processed"
        ] = (
            bloque.loc[
                mascara_vacios,
                "review_text_normalized"
            ]
            .apply(recuperar_texto_vacio)
        )

        # Sobrescribir únicamente el bloque actualizado
        bloque.to_parquet(
            archivo,
            index=False
        )

        total_textos_recuperados += vacios_bloque

    # Comprobar si queda algún texto vacío
    vacios_finales_bloque = (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    total_textos_vacios_finales += (
        vacios_finales_bloque
    )

    print(
        f"Bloque {numero_bloque}/"
        f"{len(archivos_lematizados)}: "
        f"{vacios_bloque:,} tratados, "
        f"{vacios_finales_bloque:,} vacíos restantes"
    )

Bloque 1/47: 3 tratados, 0 vacíos restantes
Bloque 2/47: 6 tratados, 0 vacíos restantes
Bloque 3/47: 7 tratados, 0 vacíos restantes
Bloque 4/47: 2 tratados, 0 vacíos restantes
Bloque 5/47: 4 tratados, 0 vacíos restantes
Bloque 6/47: 9 tratados, 1 vacíos restantes
Bloque 7/47: 6 tratados, 0 vacíos restantes
Bloque 8/47: 11 tratados, 6 vacíos restantes
Bloque 9/47: 5 tratados, 0 vacíos restantes
Bloque 10/47: 40 tratados, 29 vacíos restantes
Bloque 11/47: 5 tratados, 0 vacíos restantes
Bloque 12/47: 10 tratados, 0 vacíos restantes
Bloque 13/47: 10 tratados, 2 vacíos restantes
Bloque 14/47: 7 tratados, 0 vacíos restantes
Bloque 15/47: 1 tratados, 0 vacíos restantes
Bloque 16/47: 0 tratados, 0 vacíos restantes
Bloque 17/47: 3 tratados, 0 vacíos restantes
Bloque 18/47: 3 tratados, 0 vacíos restantes
Bloque 19/47: 1 tratados, 0 vacíos restantes
Bloque 20/47: 0 tratados, 0 vacíos restantes
Bloque 21/47: 5 tratados, 0 vacíos restantes
Bloque 22/47: 3 tratados, 0 vacíos restantes
Bloque 23/47: 

In [51]:
print("\nRESULTADO DE LA RECUPERACIÓN")
print("----------------------------")
print(
    "Textos sometidos al tratamiento alternativo:",
    f"{total_textos_recuperados:,}"
)
print(
    "Textos vacíos restantes:",
    f"{total_textos_vacios_finales:,}"
)


RESULTADO DE LA RECUPERACIÓN
----------------------------
Textos sometidos al tratamiento alternativo: 220
Textos vacíos restantes: 45


In [52]:
# Recuperación 45 restantes

resenas_vacias_finales = []

for archivo in archivos_lematizados:
    bloque = pd.read_parquet(
        archivo,
        columns=[
            "rating",
            "parent_asin",
            "review_text_normalized",
            "review_text_processed"
        ]
    )

    mascara_vacios = (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    seleccion = bloque.loc[
        mascara_vacios
    ].copy()

    if not seleccion.empty:
        resenas_vacias_finales.append(
            seleccion
        )

resenas_vacias_finales = pd.concat(
    resenas_vacias_finales,
    ignore_index=True
)

print(
    "Textos vacíos restantes:",
    f"{len(resenas_vacias_finales):,}"
)

resenas_vacias_finales

Textos vacíos restantes: 45


,rating,parent_asin,review_text_normalized,review_text_processed
0,5,B07P4J7PL7,. .,
1,5,B002RWJQ5O,- -,
2,5,B004JQFTDY,- -,
3,5,B00VMUELSO,- -,
4,5,B003DNSR3S,- -,
5,5,B003DKJAN2,- -,
6,5,B0035FZMU0,- -,
7,5,B009T3EY1A,- -,
8,5,B00ZP9XO9U,- -,
9,5,B08XQM1QHW,- -,


In [53]:
# Eliminar registros
total_eliminadas_vacias = 0
total_conservadas_final = 0

for numero_bloque, archivo in enumerate(
    archivos_lematizados,
    start=1
):
    bloque = pd.read_parquet(archivo)

    # Identificar textos procesados vacíos
    mascara_vacios = (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
    )

    eliminadas_bloque = int(
        mascara_vacios.sum()
    )

    # Conservar solo las reseñas con contenido procesado
    bloque_limpio = bloque.loc[
        ~mascara_vacios
    ].copy()

    # Sobrescribir el archivo con el bloque limpio
    bloque_limpio.to_parquet(
        archivo,
        index=False
    )

    total_eliminadas_vacias += eliminadas_bloque
    total_conservadas_final += len(bloque_limpio)

    print(
        f"Bloque {numero_bloque}/"
        f"{len(archivos_lematizados)}: "
        f"{eliminadas_bloque:,} eliminadas, "
        f"{len(bloque_limpio):,} conservadas"
    )

Bloque 1/47: 0 eliminadas, 39,175 conservadas
Bloque 2/47: 0 eliminadas, 32,192 conservadas
Bloque 3/47: 0 eliminadas, 33,664 conservadas
Bloque 4/47: 0 eliminadas, 35,123 conservadas
Bloque 5/47: 0 eliminadas, 32,876 conservadas
Bloque 6/47: 1 eliminadas, 33,926 conservadas
Bloque 7/47: 0 eliminadas, 35,786 conservadas
Bloque 8/47: 6 eliminadas, 40,786 conservadas
Bloque 9/47: 0 eliminadas, 32,918 conservadas
Bloque 10/47: 29 eliminadas, 34,528 conservadas
Bloque 11/47: 0 eliminadas, 37,276 conservadas
Bloque 12/47: 0 eliminadas, 33,484 conservadas
Bloque 13/47: 2 eliminadas, 36,178 conservadas
Bloque 14/47: 0 eliminadas, 33,400 conservadas
Bloque 15/47: 0 eliminadas, 28,976 conservadas
Bloque 16/47: 0 eliminadas, 31,491 conservadas
Bloque 17/47: 0 eliminadas, 31,264 conservadas
Bloque 18/47: 0 eliminadas, 32,306 conservadas
Bloque 19/47: 0 eliminadas, 36,465 conservadas
Bloque 20/47: 0 eliminadas, 26,334 conservadas
Bloque 21/47: 0 eliminadas, 29,921 conservadas
Bloque 22/47: 0 elimi

In [54]:
# Comprobación
print("\nRESULTADO FINAL")
print("---------------")
print(
    "Reseñas sin información textual eliminadas:",
    f"{total_eliminadas_vacias:,}"
)
print(
    "Reseñas conservadas:",
    f"{total_conservadas_final:,}"
)


RESULTADO FINAL
---------------
Reseñas sin información textual eliminadas: 45
Reseñas conservadas: 1,485,343


In [55]:
# Comprobación final
total_filas_verificadas = 0
total_vacios_verificados = 0

for archivo in archivos_lematizados:
    bloque = pd.read_parquet(
        archivo,
        columns=["review_text_processed"]
    )

    total_filas_verificadas += len(bloque)

    total_vacios_verificados += (
        bloque["review_text_processed"]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

print("\nVERIFICACIÓN FINAL")
print("------------------")
print(
    "Reseñas verificadas:",
    f"{total_filas_verificadas:,}"
)
print(
    "Textos procesados vacíos:",
    f"{total_vacios_verificados:,}"
)
print(
    "¿El resultado coincide?",
    total_filas_verificadas == 1_485_343
    and total_vacios_verificados == 0
)


VERIFICACIÓN FINAL
------------------
Reseñas verificadas: 1,485,343
Textos procesados vacíos: 0
¿El resultado coincide? True
